In [1]:
# ================================================
# Deping 프로젝트 — KOBIS API로 한국 관객수 추가
# 파일명: 02_add_kobis_audience.ipynb
# ================================================

import requests
import json
from pathlib import Path
from tqdm import tqdm
import time
from dotenv import load_dotenv
import os

# 1. .env 로드
load_dotenv()
KOBIS_API_KEY = os.getenv("KOBIS_API_KEY")

if not KOBIS_API_KEY:
    raise ValueError("❌ .env에 KOBIS_API_KEY가 없습니다! 키를 발급받아 추가해주세요.")

print("✅ KOBIS API Key 로드 완료")

# 2. 저장 폴더 생성
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 3. 기존 JSON 파일 읽기
input_file = PROCESSED_DIR / "tmdb_movies_kr_with_credits_keywords.json"

if not input_file.exists():
    raise FileNotFoundError(f"❌ {input_file} 파일이 없습니다. 01_tmdb_collect.ipynb를 먼저 실행해주세요.")

with open(input_file, "r", encoding="utf-8") as f:
    movies = json.load(f)

print(f"✅ {len(movies)}개 영화 데이터를 불러왔습니다.")

# 4. KOBIS API로 관객수 가져오는 함수
def get_kobis_audience(title_ko: str, release_date: str = None):
    """KOBIS에서 영화 검색 → 누적 관객수(audiAcc) 반환"""
    # Step 1: 영화 검색
    url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieList.json"
    params = {
        "key": KOBIS_API_KEY,
        "movieNm": title_ko,
        "itemPerPage": 10
    }
    
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        
        data = resp.json()
        movie_list = data.get("movieListResult", {}).get("movieList", [])
        
        if not movie_list:
            return None
        
        # Step 2: 개봉일이 가장 비슷한 영화 선택 (정확도 ↑)
        best_match = None
        best_score = -1
        
        for movie in movie_list:
            prdt_year = movie.get("prdtYear", "")
            if release_date and release_date.startswith(prdt_year):
                score = 100
            else:
                score = 50
            if score > best_score:
                best_score = score
                best_match = movie
        
        if not best_match:
            return None
        
        movie_cd = best_match["movieCd"]
        
        # Step 3: 영화 상세정보에서 누적 관객수 가져오기
        detail_url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json"
        detail_params = {"key": KOBIS_API_KEY, "movieCd": movie_cd}
        
        detail_resp = requests.get(detail_url, params=detail_params, timeout=10)
        if detail_resp.status_code == 200:
            detail = detail_resp.json()
            audi_acc = detail.get("movieInfoResult", {}).get("movieInfo", {}).get("audiAcc")
            return int(audi_acc) if audi_acc and audi_acc.isdigit() else None
        return None
        
    except Exception as e:
        print(f"⚠️ API 오류 ({title_ko}): {e}")
        return None

# 5. 관객수 일괄 추가
print(f"\n🚀 {len(movies)}개 영화에 대해 KOBIS 관객수 보강 시작...")

for movie in tqdm(movies, desc="KOBIS 관객수 추가"):
    if "audience_kr" not in movie:  # 이미 추가된 영화는 스킵
        audience = get_kobis_audience(
            title_ko=movie.get("title_ko"),
            release_date=movie.get("release_date")
        )
        movie["audience_kr"] = audience
        time.sleep(0.3)  # Rate limit 안전 (하루 3,000회 제한)

# 6. 결과 저장
output_file = PROCESSED_DIR / "tmdb_movies_kr_with_audience.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(movies, f, ensure_ascii=False, indent=2)

print("\n🎉 완료!")
print(f"   • 관객수가 추가된 영화 수: {sum(1 for m in movies if m.get('audience_kr') is not None)}개")
print(f"   • 저장 파일: {output_file.resolve()}")

✅ KOBIS API Key 로드 완료
✅ 240개 영화 데이터를 불러왔습니다.

🚀 240개 영화에 대해 KOBIS 관객수 보강 시작...


KOBIS 관객수 추가: 100%|██████████| 240/240 [04:23<00:00,  1.10s/it]


🎉 완료!
   • 관객수가 추가된 영화 수: 0개
   • 저장 파일: D:\deping\notebooks\data\processed\tmdb_movies_kr_with_audience.json


In [ ]:
# ==================== 단일 영화 테스트 코드 ====================
import requests
from dotenv import load_dotenv
import os

load_dotenv()
KOBIS_API_KEY = os.getenv("KOBIS_API_KEY")
print("현재 사용 중인 키:", KOBIS_API_KEY[:10] + "..." if KOBIS_API_KEY else "키 없음")

# 첫 번째 영화로 테스트 (당신 파일의 첫 영화)
test_movie = {
    "title_ko": "아바타: 불과 재",
    "release_date": "2025-12-17"
}

print(f"\n테스트 영화: {test_movie['title_ko']} ({test_movie['release_date']})")

# 1단계: 영화 검색
url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieList.json"
params = {
    "key": KOBIS_API_KEY,
    "movieNm": test_movie["title_ko"],
    "itemPerPage": 10
}

resp = requests.get(url, params=params)
print(f"1단계 검색 상태코드: {resp.status_code}")

if resp.status_code == 200:
    data = resp.json()
    movie_list = data.get("movieListResult", {}).get("movieList", [])
    print(f"검색된 영화 개수: {len(movie_list)}개")
    
    if movie_list:
        for m in movie_list[:3]:  # 상위 3개만 출력
            print(f"  - {m.get('movieNm')} ({m.get('prdtYear')}) movieCd={m.get('movieCd')}")
else:
    print("응답 내용:", resp.text[:500])

In [1]:
# ==================== 문제 확인용 디버깅 코드 ====================

import json

from pathlib import Path



file_path = Path("data/processed/tmdb_movies_kr_with_credits_keywords2.json")



with open(file_path, "r", encoding="utf-8") as f:

    movies = json.load(f)



no_audience = [m for m in movies if m.get("audience_kr") is None]

has_audience = [m for m in movies if m.get("audience_kr") is not None]



print(f"총 영화 수: {len(movies)}개")

print(f"관객수가 추가된 영화: {len(has_audience)}개")

print(f"관객수가 없는 영화: {len(no_audience)}개")



# 개봉 연도별 분포 확인

from collections import Counter

years = [m.get("year") for m in movies if m.get("year")]

print("\n개봉 연도 분포:")

for year, count in sorted(Counter(years).items()):

    print(f"   {year}년: {count}편")

    

총 영화 수: 500개
관객수가 추가된 영화: 0개
관객수가 없는 영화: 500개

개봉 연도 분포:
   1956년: 1편
   1957년: 2편
   1962년: 5편
   1969년: 1편
   1970년: 1편
   1972년: 1편
   1974년: 1편
   1975년: 2편
   1977년: 2편
   1978년: 3편
   1979년: 1편
   1982년: 1편
   1983년: 3편
   1984년: 4편
   1985년: 1편
   1986년: 1편
   1987년: 4편
   1988년: 2편
   1989년: 1편
   1990년: 2편
   1991년: 5편
   1992년: 2편
   1993년: 2편
   1994년: 6편
   1995년: 8편
   1996년: 3편
   1997년: 8편
   1998년: 8편
   1999년: 12편
   2000년: 6편
   2001년: 10편
   2002년: 9편
   2003년: 12편
   2004년: 13편
   2005년: 13편
   2006년: 9편
   2007년: 8편
   2008년: 7편
   2009년: 6편
   2010년: 10편
   2011년: 9편
   2012년: 11편
   2013년: 15편
   2014년: 18편
   2015년: 17편
   2016년: 13편
   2017년: 24편
   2018년: 18편
   2019년: 25편
   2020년: 2편
   2021년: 22편
   2022년: 23편
   2023년: 23편
   2024년: 33편
   2025년: 43편
   2026년: 8편


In [2]:
# ================================================
# Deping — 엑셀 파일에서 관객수 가져와 JSON에 추가
# ================================================

import pandas as pd
import json
from pathlib import Path

# 파일 경로 설정
PROCESSED_DIR = Path("data/processed")
excel_file = PROCESSED_DIR / "movie_posters_final.xlsx"          # ← 팀원이 준 파일
json_file = PROCESSED_DIR / "tmdb_movies_kr_with_credits_keywords.json"
output_file = PROCESSED_DIR / "tmdb_movies_kr_with_audience.json"

# 1. 엑셀 파일 읽기
print("📥 엑셀 파일 읽는 중...")
df = pd.read_excel(excel_file)

# 컬럼명 정리 (필요 시)
df = df.rename(columns={"영화명": "영화명", "누적관객수": "누적관객수"})

print(f"✅ 엑셀에서 {len(df)}개 영화 데이터 로드 완료")

# 2. 기존 JSON 파일 읽기
with open(json_file, "r", encoding="utf-8") as f:
    movies = json.load(f)

print(f"✅ JSON에서 {len(movies)}개 영화 로드 완료")

# 3. 제목 매칭해서 관객수 추가
matched_count = 0

for movie in movies:
    title_ko = movie.get("title_ko")
    if not title_ko:
        continue
    
    # 엑셀에서 같은 제목 찾기
    match = df[df["영화명"] == title_ko]
    
    if not match.empty:
        audience = match.iloc[0]["누적관객수"]
        movie["audience_kr"] = int(audience) if pd.notna(audience) else None
        matched_count += 1
    else:
        movie["audience_kr"] = None

# 4. 결과 저장
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(movies, f, ensure_ascii=False, indent=2)

print("\n🎉 작업 완료!")
print(f"   • 관객수가 성공적으로 추가된 영화: {matched_count}개")
print(f"   • 저장된 파일: {output_file.resolve()}")

📥 엑셀 파일 읽는 중...
✅ 엑셀에서 245개 영화 데이터 로드 완료
✅ JSON에서 240개 영화 로드 완료

🎉 작업 완료!
   • 관객수가 성공적으로 추가된 영화: 38개
   • 저장된 파일: D:\deping\notebooks\data\processed\tmdb_movies_kr_with_audience.json
